# Copernicus Sentinel-2 Exploration

This notebook showcases how to use the `geospatial_tools` library to search for and download Sentinel-2 products from the **Copernicus Data Space Ecosystem (CDSE)** STAC catalog. 

We will demonstrate:
1. Initializing the `StacSearch` client for Copernicus.
2. Performing a spatial and temporal search.
3. Inspecting STAC items and assets.
4. Downloading data using the **S3 Protocol** for optimized access.

In [ ]:
import os
from pathlib import Path
import leafmap
from geospatial_tools.stac import StacSearch, COPERNICUS
from geospatial_tools.copernicus.sentinel_2 import CopernicusS2Collection, CopernicusS2Band
from geospatial_tools import DATA_DIR

# The .env file is automatically loaded via geospatial_tools.__init__
print(f"Data Directory: {DATA_DIR}")

## 1. Define Search Parameters

We'll search for Sentinel-2 Level-2A data over Rome, Italy, for a specific period in 2024.

In [ ]:
# Bounding box for Rome [min_lon, min_lat, max_lon, max_lat]
bbox = [12.4, 41.8, 12.5, 41.9]
date_range = "2024-07-01/2024-07-31"
collections = [CopernicusS2Collection.L2A.value]

print(f"Searching in {bbox} for period {date_range}...")

## 2. Initialize StacSearch and Query

Initializing `StacSearch` with `catalog_name=COPERNICUS` automatically configures the S3 client using the credentials found in your `.env` file.

In [ ]:
stac_search = StacSearch(catalog_name=COPERNICUS)

results = stac_search.search(
    bbox=bbox, 
    date_range=date_range, 
    collections=collections,
    max_items=5
)

print(f"Found {len(results)} items.")

## 3. Inspect Results

Let's look at the first item and its available assets.

In [ ]:
if results:
    item = results[0]
    print(f"Item ID: {item.id}")
    print(f"Cloud Cover: {item.properties.get('eo:cloud_cover')}% ")
    
    # List a few bands/assets
    available_assets = list(item.assets.keys())
    print(f"First 10 assets: {available_assets[:10]}")

We can visualize the search area and the footprint of the first result using `leafmap`.

In [ ]:
m = leafmap.Map()
m.add_stac_item(item)
m

## 4. Download Assets using S3

Now we download the True Color Image (TCI) and the NIR band (B08) using the optimized S3 protocol.

In [ ]:
download_dir = DATA_DIR / "sentinel-2" / "copernicus_exploration"
download_dir.mkdir(parents=True, exist_ok=True)

bands = [
    CopernicusS2Band.TCI.value, 
    CopernicusS2Band.B08.value
]

print(f"Downloading bands {bands} to {download_dir} via S3...")

# download_search_results will handle the dispatcher logic automatically
downloaded_assets = stac_search.download_search_results(
    bands=bands, 
    base_directory=download_dir
)

for asset in downloaded_assets:
    asset.show_asset_items()

## Conclusion

By using the `StacSearch` dispatcher with the `COPERNICUS` catalog, you can seamlessly switch from HTTP downloads to direct S3 bucket access, which is significantly faster and more reliable for large-scale data retrieval.